In [0]:
# [Databricks Cell 2]
import os
import sys
import uuid
import json
import logging
import structlog
from contextlib import asynccontextmanager
from fastapi import FastAPI, Response, status
from pydantic import BaseModel, Field
from azure.servicebus.aio import ServiceBusClient
from azure.servicebus import ServiceBusMessage

# 1. Structured Logging Config
logging.basicConfig(format="%(message)s", stream=sys.stdout, level=logging.INFO)
structlog.configure(
    processors=[
        structlog.stdlib.add_log_level,
        structlog.processors.TimeStamper(fmt="iso"),
        structlog.processors.JSONRenderer()
    ],
    context_class=dict,
    logger_factory=structlog.stdlib.LoggerFactory(),
    wrapper_class=structlog.stdlib.BoundLogger,
    cache_logger_on_first_use=True,
)
logger = structlog.get_logger(__name__)

# 2. Asynchronous Queue Manager
class RabbitMQManager:
    def __init__(self):
        self.connection_string = os.getenv("AZURE_SERVICEBUS_CONNECTION_STRING", "Endpoint=sb://mock.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=mock=")
        self.queue_name = os.getenv("AZURE_QUEUE_NAME", "task-queue")
        self.client = None
        self.sender = None

    async def connect(self):
        if "mock=" in self.connection_string:
            logger.warning("Running with dummy keys. Queue operations will be simulated.")
            return
        self.client = ServiceBusClient.from_connection_string(conn_str=self.connection_string)
        self.sender = self.client.get_queue_sender(queue_name=self.queue_name)

    async def disconnect(self):
        if self.sender: await self.sender.close()
        if self.client: await self.client.close()

    async def send_message(self, content: str):
        if self.sender:
            await self.sender.send_messages(ServiceBusMessage(content))
        else:
            logger.info("[SIMULATION] Message queued internally", message=content)

queue_manager = RabbitMQManager()

# 3. FastAPI Initialization
@asynccontextmanager
async def lifespan(app: FastAPI):
    await queue_manager.connect()
    yield
    await queue_manager.disconnect()

app = FastAPI(title="Databricks-FastAPI-Service", lifespan=lifespan)

# 4. Endpoints
@app.get("/health")
async def health_check():
    return {"status": "healthy", "runtime": "databricks-driver"}

class TaskRequest(BaseModel):
    user_id: str
    payload: dict

@app.post("/tasks", status_code=status.HTTP_202_ACCEPTED)
async def create_task(task: TaskRequest):
    task_id = str(uuid.uuid4())
    log = logger.bind(task_id=task_id, user_id=task.user_id)
    
    message_payload = json.dumps({"task_id": task_id, "data": task.payload})
    await queue_manager.send_message(message_payload)
    
    log.info("Task successfully offloaded to message queue")
    return {"status": "accepted", "task_id": task_id}

In [0]:
import threading
import time
import uvicorn

class DatabricksBackgroundServer(uvicorn.Server):
    def install_signal_handlers(self):
        # Override to prevent conflicts with the active IPython kernel
        pass

# Configure to listen purely on localhost within the driver instance
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_config=None)
server = DatabricksBackgroundServer(config=config)

# Execute via a daemonized background thread
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

# Give the server an instant to initialize ports cleanly
time.sleep(1)
print("FastAPI background process successfully started on http://127.0.0.1:8000")

Started server process [2024]
Waiting for application startup.
{"event": "Running with dummy keys. Queue operations will be simulated.", "level": "warning", "timestamp": "2026-06-09T09:57:04.735641Z"}
Application startup complete.
Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
FastAPI background process successfully started on http://127.0.0.1:8000


In [0]:
import os
import structlog
import time
import uvicorn

class RabbitMQManager:
    def __init__(self):
        self.connection_string = os.getenv("AZURE_SERVICEBUS_CONNECTION_STRING", "Endpoint=sb://mock.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=mock=")
        self.queue_name = os.getenv("AZURE_QUEUE_NAME", "task-queue")
        self.client = None
        self.sender = None

queue_manager = RabbitMQManager()


In [0]:
import httpx
import asyncio
import random
import time

TARGET_URL = "http://127.0.0.1:8000/execute"
TEST_DURATION_SECONDS = 15  # Total test run time
TARGET_RPS = 10             # Throughput ceiling

async def send_stress_request(client: httpx.AsyncClient, request_index: int):
    """Generates and dispatches a single randomized payload to the router."""
    # Alternate between user types to stress both threads and async logic
    user_type = random.choice(["ml_user_stress", "db_user_stress"])

    payload = {
        "user_name": f"{user_type}_{request_index}_{random.randint(100, 999)}",
        "payload": {"data": {"metric_a": random.random(), "batch_id": request_index}}
    }

    start_time = time.time()
    try:
        response = await client.post(TARGET_URL, json=payload, timeout=5.0)
        latency = time.time() - start_time
        return response.status_code, latency
    except Exception as e:
        return "ERROR", 0.0

async def execute_stress_test():
    print(f"🚀 Starting stress test: {TARGET_RPS} RPS targeting {TARGET_URL}...\n")

    # Use connection pooling for accurate performance metrics
    limits = httpx.Limits(max_keepalive_connections=20, max_connections=50)
    async with httpx.AsyncClient(limits=limits) as client:

        all_latencies = []
        error_count = 0
        success_count = 0

        for second in range(TEST_DURATION_SECONDS):
            start_of_second = time.time()

            # Spawn exactly 10 requests concurrently for this slice
            tasks = [send_stress_request(client, (second * TARGET_RPS) + i) for i in range(TARGET_RPS)]
            results = await asyncio.gather(*tasks)

            # Extract statistics for this batch
            for status_code, latency in results:
                if status_code == 200:
                    success_count += 1
                    all_latencies.append(latency)
                else:
                    error_count += 1

            # Calculate execution window overhead and throttle to stay at 1 RPS rhythm
            elapsed = time.time() - start_of_second
            if elapsed < 1.0:
                await asyncio.sleep(1.0 - elapsed)

            print(f"[Second {second + 1:02d}] Dispatched {TARGET_RPS} requests. Current batch processing time: {elapsed:.3f}s")

        # Display Final Summary Report
        print("\n=== STRESS TEST RESULTS ===")
        print(f"Total Requests Dispatched : {TEST_DURATION_SECONDS * TARGET_RPS}")
        print(f"Successful Actions (200 OK): {success_count}")
        print(f"Failed/Dropped Packets    : {error_count}")
        if all_latencies:
            avg_latency = sum(all_latencies) / len(all_latencies)
            p95_latency = sorted(all_latencies)[int(len(all_latencies) * 0.95)]
            print(f"Average Server Latency    : {avg_latency * 1000:.2f} ms")
            print(f"95th Percentile Latency   : {p95_latency * 1000:.2f} ms")

# Execute the async engine loop directly inside the notebook
await execute_stress_test()

Received command c on object id p0
🚀 Starting stress test: 10 RPS targeting http://127.0.0.1:8000/execute...

127.0.0.1:51628 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Found"
127.0.0.1:51642 - "POST /execute HTTP/1.1" 404
127.0.0.1:51648 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Found"
127.0.0.1:51662 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Found"
127.0.0.1:51670 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Found"
127.0.0.1:51674 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Found"
127.0.0.1:51678 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Found"
127.0.0.1:51688 - "POST /execute HTTP/1.1" 404
HTTP Request: POST http://127.0.0.1:8000/execute "HTTP/1.1 404 Not Fou